# Machine Learning Classifier for Lung Disease

## Logistic Regression

In [ ]:
import os
import cv2
import numpy as np
from cv2 import ml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
def load_lung_disease_dataset(base_path, img_size=(224, 224)):
    data = []
    labels = []
    
    if not os.path.exists(base_path):
        print(f"Error: Path {base_path} not found.")
        return None, None
        
    class_names = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    for idx, label in enumerate(class_names):
        folder_path = os.path.join(base_path, label)
        files = os.listdir(folder_path)
        
        for img_name in tqdm(files, desc=f"Loading {label}", leave=False):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                continue
            
            img = clahe.apply(img)
            img = cv2.resize(img, img_size)
            
            data.append(img.flatten() / 255.0)
            labels.append(idx)
            
    return np.array(data, dtype=np.float32), np.array(labels, dtype=np.int32)

### Load Data set

In [ ]:
X, y = load_lung_disease_dataset('1/dataset_combined')

### Split Dataset
- Three-Way Split (Train 70%, Val 15%, Test 15%)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.176, random_state=42, stratify=y_train_full
)

y_train = y_train.reshape(-1, 1).astype(np.float32)
y_val = y_val.reshape(-1, 1).astype(np.float32)
y_test = y_test.reshape(-1, 1).astype(np.float32)

### Hyperparameter Tuning (Grid Search)

In [ ]:
learning_rates = [0.1, 0.01, 0.001]

mini_batch_sizes = [32, 64, 128]
mb_sizes = tqdm(mini_batch_sizes)

total_iters = 500

best_val_acc = -1
best_params = {}
best_model = None

print(f"\nStarting Grid Search on {len(learning_rates) * len(mini_batch_sizes)} combinations")

for lr_val in learning_rates:
    for batch_size in mb_sizes:
        mb_sizes.set_description(f"Trying {lr_val} Learning Rate, {batch_size} Batch Size")
        
        model = ml.LogisticRegression_create()
        model.setTrainMethod(ml.LogisticRegression_MINI_BATCH)
        model.setMiniBatchSize(batch_size)
        model.setLearningRate(lr_val)
        model.setIterations(total_iters)
        
        model.train(X_train, ml.ROW_SAMPLE, y_train)
        
        _, y_val_pred = model.predict(X_val)
        val_accuracy = (np.sum(y_val_pred.flatten() == y_val.flatten()) / y_val.size) * 100
        
        print(f"Tested -> LR: {lr_val}, Batch: {batch_size} | Val Acc: {val_accuracy:.2f}%")
        
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_params = {'lr': lr_val, 'batch': batch_size}
            best_model = model

print(f"\nBest Parameters: {best_params} with {best_val_acc:.2f}% Validation Accuracy.")


### Final Evaluation on Test Set


In [ ]:
print("\nEvaluating best model on unseen Test Set...")
_, y_pred = best_model.predict(X_test)

final_accuracy = (np.sum(y_pred.flatten() == y_test.flatten()) / y_test.size) * 100
print(f'Final Test Accuracy: {final_accuracy:.2f}%')

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix (LR: {best_params['lr']}, Batch: {best_params['batch']})")
plt.show()

best_model.save("best_lung_disease_model.yml")